# 04 · 선택 고정·최종 평가·보고서
A/B 3개 seed와 C의 개발검증 결과를 확인한 뒤 선택을 고정합니다. 이후에만 gate/forget/RouterArena를 평가합니다. 보호 평가에서 실패하면 배포 후보 판단을 보류하며 결과에 맞춰 다시 학습하지 않습니다.

## 공통 준비
개인 `alexyoo0506@gmail.com` Colab Pro와 해당 계정의 Drive를 사용합니다. 사용자가 Drive 5TB 보유를 확인했으므로 저장용량 확인·입력 절차는 생략합니다. Drive 연결 시 해당 계정을 선택합니다. GPU 작업은 T4 런타임을 선택하고, Drive 연결은 Google 화면에서 직접 승인합니다. GCP·Vertex 제출 또는 GCS 업로드 코드는 없습니다.

현재 파일은 아직 원격 main에 게시되지 않았으므로 main을 clone해서 실행하지 않습니다. 제공한 실행 패키지의 `transfer-experiments` 폴더를 개인 Drive의 `MyDrive/LLM_Router/`에 업로드하세요. 경로·SHA·데이터·source pin이 모두 채워져 있으므로 수정 없이 전체 실행합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

BASE = Path("/content/drive/MyDrive/LLM_Router/transfer-experiments")
SOURCE_ZIP = BASE / "source" / "transfer-source.zip"
EXPECTED_SOURCE_SHA256 = SOURCE_ZIP.with_suffix(".sha256").read_text().split()[0]
RUN_ID = "r1-to-r0-transfer-v2-run001"  # A/B/C 모두 같은 ID. 새 계획에는 새 ID
RUN = BASE / "runs" / RUN_ID
INPUTS = BASE / "inputs" / "experiment-A"
PINS = INPUTS / "source_pins.json"
DATA = INPUTS / "data"
LOG = INPUTS / "feedback-log.parquet"

assert len(EXPECTED_SOURCE_SHA256) == 64, "소스 ZIP의 검증된 SHA-256을 입력하세요"
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
REPO = Path("/content/router-transfer-source")
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, expected in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, name
EXPERIMENT = REPO / "ml/notebooks/r1_to_r0_transfer"
HELPER = EXPERIMENT / "experiment.py"
CONFIG = EXPERIMENT / "experiment_config.json"
ENV = dict(os.environ, PYTHONPATH=str(REPO / "ml/src"))
def run(*args):
    subprocess.run([sys.executable, str(HELPER), *map(str, args)], env=ENV, check=True)
print("Shared run directory:", RUN)


## 패키지 설치
Colab 메뉴 **런타임 → 런타임 유형 변경 → 런타임 버전 2026.07**(Python 3.12.13), GPU T4를 선택합니다. 기본 최신 버전은 사용하지 않습니다. Python 3.11/3.12와 CUDA 12.4용 PyTorch 2.5.1을 사용합니다. 설치 후 이미 import한 패키지 버전이 다르면 Colab 세션을 재시작하고 경로 설정 셀부터 다시 실행하세요. 학습 helper가 런타임 계약 불일치를 검사합니다.

In [ ]:
assert sys.version_info[:2] in {(3, 11), (3, 12)}, (f"현재 Python {sys.version.split()[0]}. Colab 런타임 → 런타임 유형 변경 → 런타임 버전을 2026.07(Python 3.12)로 바꾸고 처음부터 모두 실행하세요. T4는 유지합니다.")
# Text-only model: remove preinstalled optional packages incompatible with pinned torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(EXPERIMENT / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO / "ml")], check=True)


## 선택 고정
B의 8192-token 개발검증 source-macro-F1 평균이 A보다 **0.01 초과** 높을 때만 B를 고릅니다. C는 고정 참조군이며 선택 고정은 서비스 배포 승인이 아닙니다. 모든 A/B cycle-06과 C dev 예측이 없으면 실패합니다.

In [ ]:
import torch
assert torch.cuda.is_available(), "최종 평가는 GPU 런타임에서 실행하세요"
run("score", "--run-dir", RUN, "--arm", "r1_frozen", "--seed", 0, "--phase", "baseline", "--role", "dev")
run("score", "--run-dir", RUN, "--arm", "r0_continued", "--seed", 42, "--phase", "baseline", "--role", "dev")
for arm in ["r0_continued", "r1_to_r0"]:
    for seed in [42, 43, 44]:
        run("score", "--run-dir", RUN, "--arm", arm, "--seed", seed, "--phase", "cycle-06", "--role", "dev")
run("lock", "--run-dir", RUN)


## 보호 데이터 최종 추론
이 셀은 GPU가 필요합니다. 모든 모델은 고정한 final checkpoint만 사용합니다. 원본 R0의 기준 평가도 수행해 망각을 비교합니다. 출력은 seed/phase/role/길이별로 분리됩니다.

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요"
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)


In [ ]:
for role in ["gate", "forget", "routerarena"]:
    run("score", "--run-dir", RUN, "--arm", "r1_frozen", "--seed", 0, "--phase", "baseline", "--role", role, "--allow-final-evaluation")
    run("score", "--run-dir", RUN, "--arm", "r0_continued", "--seed", 42, "--phase", "baseline", "--role", role, "--allow-final-evaluation")
    for arm in ["r0_continued", "r1_to_r0"]:
        for seed in [42, 43, 44]:
            run("score", "--run-dir", RUN, "--arm", arm, "--seed", seed, "--phase", "cycle-06", "--role", role, "--allow-final-evaluation")


## 분석 자료 생성
이후 집계는 CPU로도 가능합니다. 전체/원천·카테고리 CSV, 초기·최종 비교 그림, 혼동행렬, heatmap, A/B·B/C paired bootstrap 차이를 만듭니다. 빈 결과를 추정치로 채우지 않으며 status.json이 누락 결과를 표시합니다. 평가 자료의 실제 후보 품질·비용 열이 없으면 그 성능을 만들어 내지 않습니다.

In [ ]:
run("compare", "--run-dir", RUN)
from IPython.display import display, Image, Markdown
import pandas as pd
display(pd.read_csv(RUN / "report/metrics.csv"))
display(pd.read_csv(RUN / "report/paired-comparisons.csv"))
display(Image(filename=str(RUN / "report/adaptation-curve.png")))
print((RUN / "report/status.json").read_text())
display(Markdown((RUN / "report/REPORT.md").read_text()))


## 보관
Drive의 공통 plan·모델·checkpoint·예측·report를 보존합니다. 완료 후 보고서와 작은 증거 파일만 로컬 experiments/results/r1-to-r0-transfer-v1-실행일/로 반입합니다. GCS 업로드·CURRENT 변경·자동 배포는 포함하지 않습니다.

## 마지막에 결과 하나만 전달
아래 셀이 모델 가중치를 제외한 최종 보고서·예측·실행 기록을 하나의 ZIP으로 내려받습니다. 이 파일만 대화에 첨부하면 최종 분석을 진행할 수 있습니다.

In [ ]:
import zipfile
review_zip = RUN / "results-for-review.zip"
with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in ["plan.json", "plan.sha256.json", "selection-lock.json"]:
        z.write(RUN / name, name)
    for folder in ["report", "scores"]:
        for path in sorted((RUN / folder).rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RUN))
    for path in sorted((RUN / "arms").rglob("*.json")):
        relative = path.relative_to(RUN)
        if "checkpoints" not in relative.parts and path.name in {"completed.json", "stage-contract.json", "conversion.json", "metrics.json"}:
            z.write(path, relative)
print("최종 검토용 파일:", review_zip)
from google.colab import files
files.download(str(review_zip))
